# Task 1: Universal VAE Hyperparameter Optimization (Optuna)

This notebook executes the **Optuna Hyperparameter Search** for Task 1 as detailed in **Plan 4 §3** and **Plan 6 §1.5**.

### Key Points:
- Uses SQLite persistent storage (`task1_study.db`) for multi-device portability.
- Uses **TPE sampler** and **MedianPruner** to terminate underperforming trials early.
- Runs trials with reduced epochs (15-20% of full budget) to conserve compute.
- Retrains the winning configuration on the full training schedule.
- Performs the mandatory **Skip-Connection Ablation** (Plan 6 §1.6) comparing the winning model with vs. without a single skip connection.

In [ ]:
# 1. Configuration & Optuna Settings
N_TRIALS = 25
OPTUNA_EPOCHS = 15      # Reduced epochs per trial (Plan 7 §Tasks 1 & 2b)
FULL_TRAIN_EPOCHS = 60  # Full schedule retrain for winner (Plan 6 §1.6)

STUDY_NAME = "task1_universal_vae"
STORAGE_DB = "sqlite:///task1_study.db"

USE_WANDB = bool(os.getenv("WANDB_API_KEY"))
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "genai-assignment")
SEED = 42

In [ ]:
# 2. Imports and Environment Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna
from optuna.trial import TrialState
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import PET_IMAGES_DIR, CHECKPOINTS_MANIFEST, CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_vae import ConvVAE
from src.train_loop import train_one_epoch_vae, validate_vae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
print(f"[Optuna] Storage: {STORAGE_DB}, Study: {STUDY_NAME}")

In [ ]:
# 3. Data Loading
train_dataset = PetDataset(mode="train", corruption_mode="all", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", seed=SEED)
print(f"[Data] Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 4. Optuna Objective Function
def objective(trial: optuna.Trial) -> float:
    # Hyperparameter search space (Plan 7 §Tasks 1 & 2b)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 256])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    beta = trial.suggest_float("beta", 1e-4, 1e-1, log=True)

    # Data loaders with sampled batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = ConvVAE(
        in_channels=3,
        base_channels=base_channels,
        latent_dim=latent_dim,
        dropout=dropout,
        use_skip=False
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None

    best_trial_val_loss = float("inf")

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_metrics = train_one_epoch_vae(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            alpha=alpha,
            beta=beta,
            scaler=scaler
        )
        val_metrics = validate_vae(
            model=model,
            loader=val_loader,
            device=device,
            alpha=alpha,
            beta=beta
        )
        val_loss = val_metrics["val_loss"]

        if val_loss < best_trial_val_loss:
            best_trial_val_loss = val_loss

        # Report metric for pruning
        trial.report(val_loss, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_trial_val_loss

In [ ]:
# 5. Run Optuna Study Optimization
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=pruner
)

print(f"[Optuna] Starting optimization with {N_TRIALS} trials...")
study.optimize(objective, n_trials=N_TRIALS, timeout=None)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print(f"[Optuna Study Finished]")
print(f"  Total trials: {len(study.trials)}")
print(f"  Complete: {len(complete_trials)}, Pruned: {len(pruned_trials)}")
print(f"  Best trial: #{study.best_trial.number}")
print(f"  Best val loss: {study.best_value:.4f}")
print("  Best parameters:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

In [ ]:
# 6. Save Best Config to YAML
best_config_path = CONFIGS_ROOT / "task1_best_config.yaml"
best_config_data = {
    "task": "task1_universal_vae",
    "source_notebook": "task1_vae_optuna.ipynb",
    "best_trial_number": int(study.best_trial.number),
    "best_value": float(study.best_value),
    "params": study.best_params,
}

with open(best_config_path, "w") as f:
    yaml.dump(best_config_data, f, indent=2)

print(f"[Config] Exported best hyperparameters to {best_config_path}")

In [ ]:
# 7. Retrain Winning Model on Full Schedule (Plan 6 §1.6)
p = study.best_params
batch_size = p.get("batch_size", 32)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

winner_model = ConvVAE(
    in_channels=3,
    base_channels=p["base_channels"],
    latent_dim=p["latent_dim"],
    dropout=p["dropout"],
    use_skip=False
).to(device)

optimizer = torch.optim.Adam(winner_model.parameters(), lr=p["lr"])
scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None

best_val_loss = float("inf")
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

print(f"[Full Retrain] Training winner for {FULL_TRAIN_EPOCHS} epochs...")
for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    train_metrics = train_one_epoch_vae(
        model=winner_model,
        loader=train_loader,
        optimizer=optimizer,
        device=device,
        alpha=p["alpha"],
        beta=p["beta"],
        scaler=scaler
    )
    val_metrics = validate_vae(
        model=winner_model,
        loader=val_loader,
        device=device,
        alpha=p["alpha"],
        beta=p["beta"]
    )
    val_loss = val_metrics["val_loss"]
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss

    ckpt_dict = create_checkpoint_dict(
        model=winner_model,
        optimizer=optimizer,
        epoch=epoch,
        val_loss=val_loss,
        optuna_trial_number=study.best_trial.number,
        random_seed=SEED,
        extra_metadata=p
    )
    save_checkpoint(
        checkpoint_dir=checkpoint_dir,
        prefix="task1_universal_ae_final",
        ckpt_dict=ckpt_dict,
        is_best=is_best,
        task_name="task1",
        phase_name="final",
        device_name=device_report()["device_name"]
    )

print(f"[Full Retrain Finished] Best Val Loss: {best_val_loss:.4f}")

In [ ]:
# 8. Skip-Connection Ablation Test (Plan 6 §1.6)
# Compares the winning architecture without skips vs. with a single high-resolution skip
print("--- Running Skip Connection Ablation ---")

skip_model = ConvVAE(
    in_channels=3,
    base_channels=p["base_channels"],
    latent_dim=p["latent_dim"],
    dropout=p["dropout"],
    use_skip=True  # Single skip connection enabled
).to(device)

opt_skip = torch.optim.Adam(skip_model.parameters(), lr=p["lr"])
ablation_epochs = 15

for epoch in range(1, ablation_epochs + 1):
    train_one_epoch_vae(skip_model, train_loader, opt_skip, device, alpha=p["alpha"], beta=p["beta"])

no_skip_eval = validate_vae(winner_model, val_loader, device, alpha=p["alpha"], beta=p["beta"])
skip_eval = validate_vae(skip_model, val_loader, device, alpha=p["alpha"], beta=p["beta"])

print("--- Ablation Results ---")
print(f"No-Skip Model -> Val Loss: {no_skip_eval['val_loss']:.4f}, SSIM: {no_skip_eval['val_ssim']:.4f}")
print(f"With-Skip Model -> Val Loss: {skip_eval['val_loss']:.4f}, SSIM: {skip_eval['val_ssim']:.4f}")
print("Use this quantitative comparison in your report's architecture ablation section!")